# Week 5 Task 2: Logistic Regression

This notebook demonstrates the implementation of Logistic Regression both from scratch using Gradient Descent and using the `scikit-learn` library. It covers the math behind the intuition and binary classification evaluation.

## 1. Introduction

Logistic regression is a fundamental algorithm for binary classification problems. Rather than predicting a continuous value (like Linear Regression), Logistic Regression predicts the probability that an observation belongs to a particular category.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import warnings

warnings.filterwarnings('ignore')

# Configure matplotlib for consistent plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 2. Generating the Dataset

We will create a synthetic classification dataset with 2 features to easily visualize it in a 2D plot.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate a binary classification dataset
X, y = make_classification(n_samples=200, n_features=2, n_informative=2, n_redundant=0, 
                           n_clusters_per_class=1, flip_y=0.1, random_state=42)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Visualize the generated dataset
plt.figure(figsize=(8, 5))
plt.scatter(X_train[y_train==0][:, 0], X_train[y_train==0][:, 1], color='red', label='Class 0', alpha=0.7)
plt.scatter(X_train[y_train==1][:, 0], X_train[y_train==1][:, 1], color='blue', label='Class 1', alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Synthetic Binary Classification Data')
plt.legend()
plt.show()

## 3. Math Behind Logistic Regression

### The Hypothesis function (Sigmoid)
Logistic regression transforms the linear output $z = \theta^T x$ into a probability $p$ between $0$ and $1$ using the Sigmoid (Logistic) Function:

$$h_\theta(x) = g(\theta^T x) = \frac{1}{1 + e^{-\theta^T x}}$$

If $h_\theta(x) \ge 0.5$, we predict class 1, otherwise class 0.

### Cost Function (Log Loss / Cross-Entropy)
Unlike linear regression, mean squared error results in a non-convex function for logistic regression. Instead, we use Log Loss (Cross-Entropy):

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(h_\theta(x^{(i)})) + (1 - y^{(i)}) \log(1 - h_\theta(x^{(i)})) \right]$$

### Gradient Descent
The gradient for Logistic Regression is surprisingly identical in form to Linear Regression, although the hypothesis function $h_\theta(x)$ is different:

$$\frac{\partial}{\partial \theta_j} J(\theta) = \frac{1}{m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)}) x_j^{(i)}$$

## 4. Logistic Regression from Scratch using Gradient Descent

In [ ]:
def sigmoid(z):
    # Clip z to prevent overflow in exp
    z = np.clip(z, -250, 250)
    return 1 / (1 + np.exp(-z))

def compute_cost(X, y, theta):
    m = len(y)
    h = sigmoid(X.dot(theta))
    # epsilon to avoid log(0)
    epsilon = 1e-15
    cost = (-1/m) * np.sum(y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon))
    return cost

def gradient_descent(X, y, theta, learning_rate, iterations):
    m = len(y)
    cost_history = np.zeros(iterations)
    
    for i in range(iterations):
        h = sigmoid(X.dot(theta))
        gradient = (1/m) * X.T.dot(h - y)
        theta = theta - learning_rate * gradient
        cost_history[i] = compute_cost(X, y, theta)
        
    return theta, cost_history

def predict(X, theta, threshold=0.5):
    return (sigmoid(X.dot(theta)) >= threshold).astype(int)

### Training the Custom Model

In [ ]:
# Prepare data (add bias column x0=1)
X_train_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_test_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]

# Initialize variables
theta_initial = np.zeros(X_train_b.shape[1])
learning_rate = 0.5
iterations = 1000

# Train
theta_custom, cost_history = gradient_descent(X_train_b, y_train, theta_initial, learning_rate, iterations)

print(f"Optimized Custom Theta: {theta_custom}")

# Visualize Cost History
plt.figure(figsize=(8, 4))
plt.plot(range(iterations), cost_history, 'r-')
plt.xlabel('Iterations')
plt.ylabel('Cost function J(theta)')
plt.title('Logistic Regression Cost History')
plt.show()

In [ ]:
# Calculate accuracy of the custom model
y_pred_custom = predict(X_test_b, theta_custom)
accuracy_custom = np.mean(y_pred_custom == y_test)
print(f"Custom Model Test Accuracy: {accuracy_custom * 100:.2f}%")

## 5. Logistic Regression using Scikit-Learn

Now we will use the highly optimized `scikit-learn` library to perform the same task and compare the accuracy and parameters.

In [ ]:
# Initialize and train Scikit-Learn model
sklearn_lr = LogisticRegression(C=1e9, solver='lbfgs') # C=1e9 disables regularization to match our manual approach
sklearn_lr.fit(X_train, y_train)

sklearn_theta = np.r_[sklearn_lr.intercept_[0], sklearn_lr.coef_[0]]
print(f"Scikit-Learn Theta: {sklearn_theta}")

accuracy_sklearn = sklearn_lr.score(X_test, y_test)
print(f"Scikit-Learn Model Test Accuracy: {accuracy_sklearn * 100:.2f}%")

### Decision Boundary Comparison

In [ ]:
def plot_decision_boundary(X_train, y_train, theta, ax, title):
    # Set min and max values and give it some padding
    x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
    h = 0.02
    # Generate a grid of points
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    grid_b = np.c_[np.ones((xx.ravel().shape[0], 1)), xx.ravel(), yy.ravel()]
    Z = predict(grid_b, theta)
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdBu)
    ax.scatter(X_train[y_train==0][:, 0], X_train[y_train==0][:, 1], color='red', alpha=0.8)
    ax.scatter(X_train[y_train==1][:, 0], X_train[y_train==1][:, 1], color='blue', alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_boundary(X_train, y_train, theta_custom, ax1, "Custom Logistic Regression")
plot_decision_boundary(X_train, y_train, sklearn_theta, ax2, "Scikit-Learn Logistic Regression")
plt.tight_layout()
plt.show()

## Conclusion

Both implementations successfully classify the dataset, yielding nearly identical probabilities, cost history, and accuracies. The explicit decision boundaries confirm that our custom formula operates on the same mathematical foundation as Scikit-Learn.